# Taylor Series & Approximations

## What's covered

- **Linear approximation** — replacing a function near a point with a line (or tangent plane)
- **Quadratic approximation** — adding curvature for a better fit
- **Taylor series** — pushing the idea to arbitrary order
- **Big-O remainders** — what "approximately" actually means in practice
- **Multivariable Taylor** — using the gradient and Hessian
- **Newton's method** — quadratic approximation, exactly minimized
- **Laplace approximation** preview — quadratic approximation of log-posteriors
- Where this appears in ML — Newton/L-BFGS, variational inference, error analysis


## Local linear approximation

The whole point of calculus is to replace something complicated with something simple, *near a point*. The simplest approximation is a **line** — the tangent line we built up in notebooks 1 and 2.

If `f` is differentiable at `a`, the **linear approximation** of `f` near `a` is

$$
f(x) \approx f(a) + f'(a) \cdot (x - a)
$$

Read it as: *start at the value `f(a)`, then walk along the tangent's slope.* Exact at `x = a`. Pretty good nearby. Wrong by an amount that grows as you stray from `a`.

How wrong? For a twice-differentiable `f`, the error is bounded by a constant times `(x - a)^2`. So as `x → a`, the error goes to zero **quadratically** — much faster than linearly. This is why the linear approximation is the right local picture: small perturbations produce *very* small errors.

In ML language: when you take a gradient-descent step, the loss changes by approximately `∇L · Δw` to first order — the linear approximation. The error of that estimate is order `||Δw||^2`. That's why small learning rates work: the linear prediction is reliable.


In [ ]:
import numpy as np

# Linear approximation of e^x at a = 0: 1 + x
f      = lambda x: np.exp(x)
linear = lambda x: 1 + x   # f(0) + f'(0)(x - 0)

print(f"{'x':>6}  {'e^x':>10}  {'linear approx':>14}  {'error':>10}")
for x in [-0.5, -0.1, -0.01, 0.01, 0.1, 0.5]:
    print(f"{x:>6}  {f(x):>10.6f}  {linear(x):>14.6f}  {abs(f(x) - linear(x)):>10.2e}")

print("\nNotice: error halves twice when x shrinks by 2 — that's the quadratic decay.")


## Quadratic approximation

Add one more term to capture curvature. The **quadratic approximation** of `f` near `a` is

$$
f(x) \approx f(a) + f'(a)(x - a) + \frac{1}{2} f''(a) (x - a)^2
$$

You can read this off `e^x` at `a = 0`: `f(0) = 1`, `f'(0) = 1`, `f''(0) = 1`. So `e^x ≈ 1 + x + x^2 / 2` near 0.

The error of the quadratic approximation is `O((x - a)^3)` — even more forgiving than the linear case. Geometrically: a *parabola* hugs the curve better than a *line* at any twice-differentiable point.

For ML, the quadratic approximation is **the** local picture used by second-order optimization methods. Near a candidate minimum, a smooth loss looks like a bowl — and the bowl is described by its gradient (zero at the minimum) and its Hessian (the curvature of the walls). Newton's method, BFGS, L-BFGS, and natural gradient all minimize the *quadratic approximation* exactly at each step.


In [ ]:
# Quadratic approximation of e^x at 0: 1 + x + x^2/2
quadratic = lambda x: 1 + x + 0.5 * x ** 2

print(f"{'x':>6}  {'e^x':>10}  {'linear':>10}  {'quadratic':>11}")
for x in [-0.5, -0.1, 0.0, 0.1, 0.5, 1.0]:
    print(f"{x:>6}  {f(x):>10.6f}  {1+x:>10.6f}  {quadratic(x):>11.6f}")

print("\nNotice how the quadratic stays much closer to e^x as |x| grows.")


## Taylor series — all orders

Push the idea to all derivatives at once. The **Taylor series** of `f` about `a` is the infinite polynomial:

$$
f(x) = \sum_{n = 0}^{\infty} \frac{f^{(n)}(a)}{n!} (x - a)^n = f(a) + f'(a)(x - a) + \frac{f''(a)}{2!}(x - a)^2 + \frac{f'''(a)}{3!}(x - a)^3 + \dots
$$

When this series **converges** to `f`, we say `f` is **analytic** at `a` — and the series *is* `f`, not just an approximation. Polynomials, `e^x`, `sin x`, `cos x`, `log(1 + x)` (within radius 1), and most functions ML uses are analytic in regions you care about.

Three classic series, all centered at `a = 0`:

$$
e^x = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + \dots
$$

$$
\sin x = x - \frac{x^3}{3!} + \frac{x^5}{5!} - \frac{x^7}{7!} + \dots
$$

$$
\frac{1}{1 - x} = 1 + x + x^2 + x^3 + \dots \qquad (|x| < 1)
$$

**Truncation.** Drop terms past order `n` and you have the **`n`-th order Taylor polynomial** of `f`. The remainder (error of the truncation) is `O((x - a)^{n+1})`. Writing the remainder cleanly:

$$
f(x) = \underbrace{\sum_{k=0}^{n} \frac{f^{(k)}(a)}{k!}(x - a)^k}_{\text{polynomial}} + \underbrace{R_n(x)}_{\text{remainder, } O((x-a)^{n+1})}
$$

For ML: linear approximation is `n = 1`, quadratic is `n = 2`. We almost never go past `n = 2` because second derivatives (Hessian) are already expensive, and third derivatives are essentially never used.


In [ ]:
# Approximating sin(x) with successive Taylor polynomials at x = 0
def taylor_sin(x, terms):
    # sin(x) = x - x^3/3! + x^5/5! - ...
    result = 0.0
    sign = 1
    factorial = 1
    for k in range(terms):
        power = 2 * k + 1
        if k > 0:
            factorial *= (power - 1) * power
        result += sign * (x ** power) / factorial
        sign *= -1
    return result

x = 1.0
true_val = np.sin(x)
print(f"sin({x}) = {true_val:.10f}\n")
print(f"{'terms':<8} {'approx':>14} {'error':>12}")
for n_terms in [1, 2, 3, 4, 5, 6]:
    approx = taylor_sin(x, n_terms)
    print(f"{n_terms:<8} {approx:>14.10f} {abs(approx - true_val):>12.2e}")

print("\nEach extra term shaves the error by roughly x^2 / ((2k)(2k+1)) — very fast convergence near 0.")


## Multivariable Taylor

Lift everything to many variables. For `f: R^n → R`, expanding about a point `\mathbf{a}`:

$$
f(\mathbf{x}) \approx f(\mathbf{a}) + \nabla f(\mathbf{a})^T (\mathbf{x} - \mathbf{a}) + \frac{1}{2} (\mathbf{x} - \mathbf{a})^T H_f(\mathbf{a}) (\mathbf{x} - \mathbf{a})
$$

The roles transfer exactly: the **gradient** plays the part of `f'(a)`; the **Hessian** plays the part of `f''(a)`; the squared term `(x - a)^2` becomes the quadratic form `(x - a)^T H (x - a)`.

**Three orders, three uses:**

- **Order 0** (`f(x) ≈ f(a)`) — useless on its own, but it's where "loss landscape" intuition starts.
- **Order 1** (linear) — what every gradient-descent step assumes locally. The first-order Taylor polynomial of the loss at `w` says `L(w + Δ) ≈ L(w) + ∇L(w)^T Δ`. Minimize over `Δ` of unit length and you get `Δ = -∇L / ||∇L||` — *steepest descent*.
- **Order 2** (quadratic) — what Newton's method assumes. The second-order Taylor polynomial is itself a quadratic in `Δ`, and we can minimize it in closed form.

That last point is the heart of the next section.


## Newton's method — quadratic approximation, minimized exactly

Near a candidate point `x`, write the second-order Taylor expansion of `f`:

$$
f(\mathbf{x} + \Delta) \approx f(\mathbf{x}) + \nabla f(\mathbf{x})^T \Delta + \frac{1}{2} \Delta^T H_f(\mathbf{x}) \Delta
$$

This right-hand side is a *quadratic* in `Δ`. Minimize it: take its gradient with respect to `Δ` (using the matrix-calculus identities from notebook 5!), set to zero:

$$
\nabla f(\mathbf{x}) + H_f(\mathbf{x}) \Delta = \mathbf{0} \quad \Longrightarrow \quad \Delta = -H_f(\mathbf{x})^{-1} \nabla f(\mathbf{x})
$$

That's **Newton's step**. Move to `x + Δ`, repeat. The update rule:

$$
\boxed{\mathbf{x}_{t+1} = \mathbf{x}_t - H_f(\mathbf{x}_t)^{-1} \nabla f(\mathbf{x}_t)}
$$

**Why it's powerful.** Near a minimum (where the function genuinely is approximately quadratic), Newton's method converges *quadratically* — the error squares each iteration. From `1e-3` to `1e-6` to `1e-12` in three steps. Gradient descent only converges linearly.

**Why it's hard in ML.** Forming and inverting `H_f` is `O(n^3)` and `O(n^2)` memory — infeasible for millions of parameters. So ML uses cheap approximations: BFGS / L-BFGS (low-memory rank-1 updates), natural gradient (Fisher information as a Hessian surrogate), and Hessian-free methods (compute `H v` for a single vector `v` instead of forming `H`). All notebook 7 fare.

A note: Newton's method finds **stationary points**, not minima specifically. It can converge to saddles or maxima too. For ML, that's actually a feature in some contexts — second-order methods can escape saddles that gradient descent gets stuck near.


In [ ]:
# Newton's method on the convex quadratic f(x, y) = x^2 + 2 y^2 + x y - 4 x - 5 y
# Analytical minimum: solve ∇f = 0 → (2x + y - 4, 4y + x - 5) = 0
def f(xy):
    x, y = xy
    return x**2 + 2*y**2 + x*y - 4*x - 5*y

def grad(xy):
    x, y = xy
    return np.array([2*x + y - 4, 4*y + x - 5])

H = np.array([[2.0, 1.0],
              [1.0, 4.0]])     # Hessian — constant for a quadratic

# Start somewhere far from the optimum
xy = np.array([10.0, -3.0])
print(f"start:   xy = {xy},  f = {f(xy):.4f}")
for step in range(5):
    g = grad(xy)
    xy = xy - np.linalg.solve(H, g)
    print(f"step {step+1}: xy = {xy.round(6)},  f = {f(xy):.6f}, ||grad|| = {np.linalg.norm(grad(xy)):.2e}")

print("\nFor a quadratic, Newton's method converges in ONE step from any starting point — because the second-order approximation is exact.")


## Laplace approximation — a preview

Here's a beautiful application of quadratic Taylor that we won't fully cover until the probability repo, but is worth seeding now.

**Setup.** You have a probability distribution `p(\theta)` you can't compute analytically — say a posterior in Bayesian inference. But you *can* find its mode `\theta^*` (the MAP estimate) and you can compute its log and its Hessian.

**Idea.** Approximate `p(\theta)` by a **Gaussian** centered at `\theta^*`. Why a Gaussian? Because near its mode, `\log p(\theta)` is locally quadratic, by Taylor:

$$
\log p(\theta) \approx \log p(\theta^*) - \frac{1}{2} (\theta - \theta^*)^T \Lambda (\theta - \theta^*)
$$

where `Λ = -H_{\log p}(\theta^*)` is the negative Hessian of the log-density at the mode. Exponentiating, this is *exactly* the form of a Gaussian with mean `\theta^*` and covariance `Λ^{-1}`. So:

$$
p(\theta) \approx \mathcal{N}(\theta^*, \Lambda^{-1})
$$

This is the **Laplace approximation**. It is one of the cheapest, most-used recipes in Bayesian deep learning, integrated information theory, and approximate inference. Once you see it, you'll see it everywhere — including underneath Newton's method, BIC model selection, and many parts of variational inference.

The point for this notebook: **all of these "approximate a complicated function by a quadratic" methods are Taylor expansion in disguise.**


## Where this appears in ML

Taylor expansion is the math behind every "we'll approximate the loss locally" argument in machine learning.

- **Steepest descent.** Minimizing the first-order Taylor approximation gives `-∇L / ||∇L||` — the steepest descent direction.
- **Newton's method, BFGS, L-BFGS.** Minimize the second-order Taylor approximation. BFGS and L-BFGS approximate the Hessian cheaply with rank-1 updates — they show up in `scipy.optimize` and in classical convex-optimization solvers used inside many ML pipelines.
- **Natural gradient.** Replace the Hessian with the Fisher information matrix — gives steps that are invariant to reparameterization.
- **Trust-region methods.** Trust the second-order Taylor only within a small radius; shrink the radius when the approximation overshoots.
- **Laplace approximation.** Quadratic Taylor of `log p(θ)` at the mode gives a Gaussian. Used in Bayesian neural networks, BIC model selection, and post-hoc uncertainty quantification.
- **Variational inference.** Approximate intractable posteriors with simpler families. The ELBO bound has Taylor approximations baked into many derivations.
- **Loss-landscape analysis.** Sharpness measures (e.g., SAM, Sharpness-Aware Minimization) literally measure the quadratic part of the local Taylor expansion of `L`.
- **Gradient checkpointing math.** The compute-memory trade-off uses Taylor remainders to bound the error of recomputed activations.
- **Network expansion / pruning.** Compute "importance" of weights via first or second-order Taylor estimates of how much removing them perturbs the loss.
- **Numerical differentiation error bounds.** Forward differences have `O(h)` error from the first dropped Taylor term; central differences have `O(h^2)` from the second.
- **Diffusion models.** The Tweedie formula and Stein score estimation can be derived through Taylor expansions of log-densities.

Next notebook: **optimization and convexity** — we put the Taylor machinery to work, study gradient descent variants properly, derive convexity tests via the Hessian, and meet saddle points (which torment high-dimensional optimization).
